## 01 - Load & QC

Veri yukleme, yapisal butunluk, zaman/sinyal dogrulama, QC bayraklari.

Veri Google Drive klasorunden cekilir. Klasor link ile herkese acik oldugu
icin kimlik dogrulama yok: API anahtari, OAuth, client_secrets.json gerekmez.

In [1]:
%pip install -q pyyaml pandas numpy pyarrow gdown

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ANALYSIS_ROOT = Path.cwd()
if not (ANALYSIS_ROOT / "config.yaml").exists():
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src.drive_sync import sync_data
from src.loader import load_all
from src.qc import (
    check_structural_integrity,
    check_timing,
    check_signals,
    flag_trials,
    add_analysis_mask,
    check_randomization,
    check_format_regression,
)

with open(ANALYSIS_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

RAW_DIR = ANALYSIS_ROOT / config["paths"]["raw_dir"]
INTERIM_DIR = ANALYSIS_ROOT / config["paths"]["interim_dir"]
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
print(f"Kok: {ANALYSIS_ROOT}")

Kok: c:\Users\elifa\Documents\GitHub\NOROM_Inv_Pendulum\Data Analysis


### 0. Drive'dan veri cek

In [3]:
# Var olan dosyalar tekrar indirilmez. Yeni katilimci geldiginde
# bu hucreyi tekrar calistirmak yeterli.
sync_data(config["drive"]["folder_id"], RAW_DIR);

Drive klasoru listeleniyor...
36 dosya bulundu.
  indiriliyor: P008/S20260827_134546/P008_S20260827_134546_metadata.json
  indiriliyor: P008/S20260827_134546/P008_S20260827_134546_timeseries.csv
  indiriliyor: P008/S20260827_134546/P008_S20260827_134546_trial_summary.csv
  indiriliyor: P009/S20260827_141610/P009_S20260827_141610_metadata.json
  indiriliyor: P009/S20260827_141610/P009_S20260827_141610_timeseries.csv
  indiriliyor: P009/S20260827_141610/P009_S20260827_141610_trial_summary.csv
  indiriliyor: P010/S20260827_144609/P010_S20260827_144609_metadata.json
  indiriliyor: P010/S20260827_144609/P010_S20260827_144609_timeseries.csv
  indiriliyor: P010/S20260827_144609/P010_S20260827_144609_trial_summary.csv
  indiriliyor: P011/S20260827_151041/P011_S20260827_151041_metadata.json
  indiriliyor: P011/S20260827_151041/P011_S20260827_151041_timeseries.csv
  indiriliyor: P011/S20260827_151041/P011_S20260827_151041_trial_summary.csv
  indiriliyor: P012/S20260827_153744/P012_S20260827_1537

### 1. Kesif ve yukleme

In [4]:
df_samples, df_trials, metadata, report = load_all(RAW_DIR)

print(f"Oturumlar: {len(report)}")
n_sel = sum(1 for s in report if s["status"] == "selected")
n_inc = sum(1 for s in report if s["status"] == "incomplete")
print(f"  Secilen:   {n_sel}")
print(f"  Yarim:     {n_inc}")
print(f"Sample:      {len(df_samples):,}")
print(f"Trial:       {len(df_trials)}")
print(f"Metadata:    {len(metadata)} dosya")

if report:
    cols = [
        "participant_id", "session_id", "status",
        "has_metadata", "has_timeseries", "has_trial_summary",
        "measurement_trial_count", "warnings",
    ]
    display(pd.DataFrame(report)[cols])

Oturumlar: 12
  Secilen:   12
  Yarim:     0
Sample:      846,060
Trial:       636
Metadata:    12 dosya


,participant_id,session_id,status,has_metadata,has_timeseries,has_trial_summary,measurement_trial_count,warnings
0,P001,S20260826_151348,selected,True,True,True,50,[]
1,P002,S20260827_094922,selected,True,True,True,50,[]
2,P003,S20260827_101526,selected,True,True,True,50,[]
3,P004,S20260827_104146,selected,True,True,True,50,[]
4,P005,S20260827_110629,selected,True,True,True,50,[]
5,P006,S20260827_113330,selected,True,True,True,50,[]
6,P007,S20260827_125639,selected,True,True,True,50,[]
7,P008,S20260827_134546,selected,True,True,True,50,[]
8,P009,S20260827_141610,selected,True,True,True,50,[]
9,P010,S20260827_144609,selected,True,True,True,50,[]


### 2. Yapisal butunluk

In [5]:
issues = check_structural_integrity(df_trials, metadata, config)

if issues:
    df_iss = pd.DataFrame(issues)
    n_fail = int((df_iss["status"] == "FAIL").sum())
    n_warn = len(df_iss) - n_fail
    print(f"{len(df_iss)} sorun  (FAIL: {n_fail}, WARN: {n_warn})")
    display(df_iss.sort_values("status"))
else:
    print("Yapisal sorun yok.")

13 sorun  (FAIL: 0, WARN: 13)


,participant_id,check,status,detail
0,"P001, P002, P003, P004, P005, P006, P007, P008...",shared_condition_order,WARN,12 katilimci ayni kosul sirasini paylasiyor
1,"P001, P002, P003, P004, P005, P006, P007, P008...",shared_randomization_seed,WARN,"randomizationSeed=12345, 12 katilimcida ayni"
2,P002,config_participant_id,WARN,"config.participantId=P001, klasor=P002"
3,P003,config_participant_id,WARN,"config.participantId=P001, klasor=P003"
4,P004,config_participant_id,WARN,"config.participantId=P001, klasor=P004"
5,P005,config_participant_id,WARN,"config.participantId=P001, klasor=P005"
6,P006,config_participant_id,WARN,"config.participantId=P001, klasor=P006"
7,P007,config_participant_id,WARN,"config.participantId=P001, klasor=P007"
8,P008,config_participant_id,WARN,"config.participantId=P001, klasor=P008"
9,P009,config_participant_id,WARN,"config.participantId=P001, klasor=P009"


### 3. Zaman ve ornekleme

In [6]:
timing = check_timing(df_samples, config)

if timing.empty:
    print("Veri yok, atlaniyor.")
else:
    problem_mask = (
        timing["has_time_reversal"]
        | timing["has_gap"]
        | timing["has_dup_index"]
        | timing["has_skip_index"]
        | (timing["nan_count"] > 0)
        | (timing["angle_out_of_range"] > 0)
    )
    problems = timing[problem_mask]
    if len(problems) > 0:
        n_prob = len(problems)
        print(f"{n_prob} trial'da zaman/ornekleme sorunu:")
        display(problems)
    else:
        print("Zaman ve ornekleme sorunsuz.")

    dt_avg = timing["dt_mean"].mean()
    dt_std_avg = timing["dt_std"].mean()
    dt_max = timing["dt_max_dev"].max()
    print()
    print(f"dt ort: {dt_avg:.6f} s")
    print(f"dt std ort: {dt_std_avg:.8f} s")
    print(f"dt max sapma: {dt_max:.6f} s")

Zaman ve ornekleme sorunsuz.

dt ort: 0.016667 s
dt std ort: 0.00004715 s
dt max sapma: 0.000067 s


### 4. Sinyal akil sagligi

In [7]:
signals = check_signals(df_samples, config)

if signals.empty:
    print("Veri yok, atlaniyor.")
else:
    warn_thr = config["qc"]["velocity_correlation_warn"]

    low_corr = signals[
        (signals["cart_vel_corr"] < warn_thr)
        | (signals["omega_corr"] < warn_thr)
    ]
    if len(low_corr) > 0:
        print(f"{len(low_corr)} trial'da hiz-pozisyon korelasyonu < {warn_thr}")
        display(low_corr[["participant_id", "trial_id", "n_segments", "cart_vel_corr", "omega_corr"]])
    else:
        print(f"Hiz-pozisyon korelasyonu tum triallarda >= {warn_thr}")

    force_bad = signals[~signals["force_ok"]]
    if len(force_bad) > 0:
        print()
        print(f"{len(force_bad)} trial'da force tutarsiz:")
        display(force_bad[["participant_id", "trial_id"]])
    else:
        max_f = config["physics"]["max_force_n"]
        print()
        print(f"Force tutarliligi OK (input_applied x {max_f} N)")

    phase_bad = signals[~signals["phase_reset_ok"]]
    if len(phase_bad) > 0:
        print()
        print(f"{len(phase_bad)} trial'da phase/is_resetting tutarsiz:")
        display(phase_bad[["participant_id", "trial_id"]])
    else:
        print()
        print("phase ve is_resetting tutarli.")

Hiz-pozisyon korelasyonu tum triallarda >= 0.99

Force tutarliligi OK (input_applied x 4.0 N)

phase ve is_resetting tutarli.


### 5. Trial gecerliligi ve sample maskesi

In [8]:
df_trials = flag_trials(df_trials, df_samples, config)
df_samples = add_analysis_mask(df_samples, df_trials)

if df_trials.empty:
    print("Veri yok, atlaniyor.")
else:
    n_pass = int(df_trials["qc_pass"].sum())
    n_fail = len(df_trials) - n_pass
    print(f"Trial: {len(df_trials)}  (pass: {n_pass}, fail: {n_fail})")

    if n_fail > 0:
        print()
        print("Dusen triallar:")
        fail_cols = ["participant_id", "trial_id", "noise_level_id", "practice", "qc_flags"]
        display(df_trials[~df_trials["qc_pass"]][fail_cols])

    n_inc = int(df_samples["analysis_include"].sum())
    n_exc = len(df_samples) - n_inc
    print()
    print(f"Sample: {len(df_samples):,}  (include: {n_inc:,}, exclude: {n_exc:,})")

    meas = df_trials[(df_trials["practice"] == 0) & df_trials["qc_pass"]]
    print()
    print(f"Analize giren measurement trial: {len(meas)}")
    for pid in sorted(meas["participant_id"].unique()):
        n = int((meas["participant_id"] == pid).sum())
        print(f"  {pid}: {n}")

Trial: 636  (pass: 636, fail: 0)

Sample: 846,060  (include: 720,000, exclude: 126,060)

Analize giren measurement trial: 600
  P001: 50
  P002: 50
  P003: 50
  P004: 50
  P005: 50
  P006: 50
  P007: 50
  P008: 50
  P009: 50
  P010: 50
  P011: 50
  P012: 50


## 7. Randomizasyon dogrulamasi

metadata'daki `condition_order` bir iddia. Burada once trial_summary'den
okunan gercek diziyle karsilastirilir, sonra katilimcilar arasi ozdeslik
timeseries ve trial_summary'den olculur.

In [9]:
rnd = check_randomization(df_trials, df_samples, metadata, config)

mvd = rnd["metadata_vs_data"]
if mvd.empty:
    print("Metadata yok, karsilastirilamadi.")
else:
    n_ok = int(mvd["match"].sum())
    print(f"metadata condition_order vs trial_summary: {n_ok}/{len(mvd)} katilimcida birebir")
    if n_ok < len(mvd):
        display(mvd[~mvd["match"]])

acr = rnd["across_participants"]
if not acr.empty:
    print()
    print("Katilimcilar arasi ozdeslik:")
    display(acr)

pos = rnd["position_table"]
if not pos.empty:
    n_cell = pos.values.sum() / pos.size
    print()
    print(f"Tur ici pozisyon dagilimi (hucre basina beklenen ~{n_cell:.1f}):")
    print("Betimleyici, kusur testi degil. Tur sayisi az, sapmalar sansa girebilir.")
    display(pos)

tor = rnd["trial_order_table"]
if not tor.empty:
    print()
    print("Kosul x trial_order - ogrenme/yorgunluk koşulla karismis mi:")
    display(tor)

metadata condition_order vs trial_summary: 12/12 katilimcida birebir

Katilimcilar arasi ozdeslik:


,olcut,farkli_dizi_sayisi,tum_katilimcilarda_ozdes,detay
0,kosul sirasi,1.0,True,12 katilimci
1,noise_seed dizisi,1.0,True,12 katilimci
2,baslangic acisi,NaN,False,0/53 trial'da tum katilimcilarda ayni



Tur ici pozisyon dagilimi (hucre basina beklenen ~2.0):
Betimleyici, kusur testi degil. Tur sayisi az, sapmalar sansa girebilir.


,poz_1,poz_2,poz_3,poz_4,poz_5
noise_level_id,,,,,
N1,3,1,1,2,3
N2,0,1,2,3,4
N3,1,4,0,3,2
N4,5,1,2,2,0
no_noise,1,3,5,0,1



Kosul x trial_order - ogrenme/yorgunluk koşulla karismis mi:


,mean,min,max
noise_level_id,,,
N1,25.6,1,47
N2,26.5,5,50
N3,25.6,4,46
N4,24.6,3,49
no_noise,25.2,2,48


## 8. Format regresyon takibi

In [10]:
regression = check_format_regression(metadata, config)

if regression:
    df_reg = pd.DataFrame(regression)
    present = int(df_reg["present"].sum())
    missing = len(df_reg) - present
    print(f"Istenen alanlar: {len(df_reg)}  (mevcut: {present}, eksik: {missing})")
    if missing > 0:
        print()
        print("Eksik alanlar:")
        display(df_reg[~df_reg["present"]])
else:
    print("Metadata yok, kontrol yapilamadi.")

Istenen alanlar: 24  (mevcut: 0, eksik: 24)

Eksik alanlar:


,field,present
0,screen_width_px,False
1,screen_height_px,False
2,screen_physical_width_cm,False
3,viewing_distance_cm,False
4,refresh_rate_hz,False
5,full_screen,False
6,input_axis_name,False
7,deadzone,False
8,sensitivity,False
9,invert_axis,False


### Cikti

In [11]:
if df_samples.empty or df_trials.empty:
    print("Veri yok, cikti yazilmiyor.")
else:
    df_samples.to_parquet(INTERIM_DIR / "samples_clean.parquet", index=False)
    df_trials.to_parquet(INTERIM_DIR / "trials_clean.parquet", index=False)
    print(f"samples_clean.parquet  ({len(df_samples):,} satir)")
    print(f"trials_clean.parquet   ({len(df_trials)} satir)")

samples_clean.parquet  (846,060 satir)
trials_clean.parquet   (636 satir)
